In [8]:
import numpy as np
import matplotlib.pyplot as plt
import os
from astropy.io import fits
from astropy.wcs import WCS
import astroalign as aa

# Fix F9 alignment

In [13]:
# -----------------------------
# User inputs
# -----------------------------

# Continuum images used to MEASURE the transforms
sn3_reference_continuum = "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN3Continuum.fits"
sn2_continuum_for_alignment = "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN2Continuum.fits"
sn1_continuum_for_alignment = "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN1Continuum.fits"

# Files to align with the SN2 transform
sn2_files_to_align = [
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-Hbflux.fits",
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-Hbflux-err.fits",
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-OIII5007flux.fits",
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-OIII5007flux-err.fits",
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN2Continuum.fits",
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN2Continuum-err.fits",
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN2Deep.fits",
]

# Files to align with the SN1 transform
sn1_files_to_align = [
    "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-OII3727flux.fits",
     "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-OII3727flux-err.fits",
     "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN1Continuum.fits",
     "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN1Continuum-err.fits",
     "../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN1Deep.fits",
]

# Output directory
output_dir = "../M33-Maps-Calibrated/M33-F9"


# -----------------------------
# Helpers
# -----------------------------

def load_fits_data_header(filename):
    with fits.open(filename) as hdul:
        data = hdul[0].data.astype(float)
        header = hdul[0].header.copy()
    return data, header


def ensure_2d(data, filename):
    if data.ndim != 2:
        raise ValueError(
            f"{filename} has shape {data.shape}. "
            "This script expects 2D FITS images/maps."
        )
    return data


def safe_for_alignment(data):
    """
    Prepare image for astroalign star matching.
    Keeps original data untouched for final resampling.
    """
    arr = np.array(data, dtype=float, copy=True)

    bad = ~np.isfinite(arr)
    if np.all(bad):
        raise ValueError("Image contains only NaN/Inf values.")

    # Replace bad pixels with median of valid values
    med = np.nanmedian(arr)
    arr[bad] = med

    return arr


def compute_transform(source_continuum_file, reference_continuum_file):
    """
    Measure the transform that maps source -> reference.
    """
    ref_data, ref_header = load_fits_data_header(reference_continuum_file)
    src_data, src_header = load_fits_data_header(source_continuum_file)

    ref_data = ensure_2d(ref_data, reference_continuum_file)
    src_data = ensure_2d(src_data, source_continuum_file)

    ref_for_match = safe_for_alignment(ref_data)
    src_for_match = safe_for_alignment(src_data)

    transform, (src_pts, ref_pts) = aa.find_transform(src_for_match, ref_for_match)

    print(f"\nTransform measured for: {os.path.basename(source_continuum_file)}")
    print(f"  matched stars: {len(src_pts)}")
    print(f"  translation: {transform.translation}")
    print(f"  rotation (rad): {transform.rotation}")
    print(f"  scale: {transform.scale}")

    return transform, ref_data, ref_header


def align_one_file(input_file, output_file, transform, reference_data, reference_header):
    """
    Apply a precomputed astroalign transform to one FITS image.
    Output is written on the reference pixel grid with the reference header.
    """
    data, header = load_fits_data_header(input_file)
    data = ensure_2d(data, input_file)

    aligned_data, footprint = aa.apply_transform(
        transform,
        data,
        reference_data,
        fill_value=np.nan
    )

    out_header = reference_header.copy()

    # Optional provenance
    out_header["HISTORY"] = f"Aligned to {os.path.basename(sn3_reference_continuum)} using astroalign"
    out_header["HISTORY"] = f"Source file: {os.path.basename(input_file)}"

    fits.PrimaryHDU(data=aligned_data, header=out_header).writeto(output_file, overwrite=True)
    print(f"  wrote: {output_file}")


def align_file_list(file_list, transform, reference_data, reference_header):
    # if len(file_list) == 0:
    #     print(f"\nNo files provided for {tag}.")
    #     return

    # tag_dir = os.path.join(output_dir, tag)
    # os.makedirs(tag_dir, exist_ok=True)

    print(f"\nAligning {len(file_list)} files")

    for input_file in file_list:
        base = os.path.basename(input_file)
        root, ext = os.path.splitext(base)
        output_file = os.path.join(output_dir, f"{root}{ext}")
        align_one_file(input_file, output_file, transform, reference_data, reference_header)


# -----------------------------
# Main
# -----------------------------

def main():
    os.makedirs(output_dir, exist_ok=True)

    # Measure SN2 -> SN3 transform
    print("Computing SN2 -> SN3 transform...")
    sn2_transform, ref_data, ref_header = compute_transform(
        sn2_continuum_for_alignment,
        sn3_reference_continuum
    )

    # Measure SN1 -> SN3 transform
    print("\nComputing SN1 -> SN3 transform...")
    sn1_transform, _, _ = compute_transform(
        sn1_continuum_for_alignment,
        sn3_reference_continuum
    )

    # Apply transforms
    align_file_list(
        sn2_files_to_align,
        sn2_transform,
        ref_data,
        ref_header    )

    align_file_list(
        sn1_files_to_align,
        sn1_transform,
        ref_data,
        ref_header    )

    print("\nDone.")


if __name__ == "__main__":
    main()

Computing SN2 -> SN3 transform...

Transform measured for: M33F9-SN2Continuum.fits
  matched stars: 15
  translation: [-150.09684687  239.56422378]
  rotation (rad): -0.04939253579218882
  scale: 1.0283181093950018

Computing SN1 -> SN3 transform...

Transform measured for: M33F9-SN1Continuum.fits
  matched stars: 9
  translation: [-146.80176523  241.51289629]
  rotation (rad): -0.04912481692335312
  scale: 1.0254031617872643

Aligning 7 files
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-Hbflux.fits
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-Hbflux-err.fits
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-OIII5007flux.fits
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-OIII5007flux-err.fits
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-SN2Continuum.fits
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-SN2Continuum-err.fits
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-SN2Deep.fits

Aligning 5 files
  wrote: ../M33-Maps-Calibrated/M33-F9/M33F9-OII3727flux.fits
  wrote: ../M33-Maps-Calibrated/M33-

In [7]:
# Fix alignment for field 9 separately where the WCS seems wrong and reprojection doesn't work


def load_fits(fn):
    with fits.open(fn) as hdul:
        data = hdul[0].data.astype(float)
        header = hdul[0].header
    return data, header

# Load your continuum images
ref_data, ref_header = load_fits("../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN3Continuum.fits")
sn2_data, sn2_header = load_fits("../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN2Continuum.fits")
sn1_data, sn1_header = load_fits("../M33-Maps-Calibrated/M33-F9-not-aligned/M33F9-SN1Continuum.fits")

# Replace NaNs for alignment only
ref = np.nan_to_num(ref_data, nan=0.0)
sn2 = np.nan_to_num(sn2_data, nan=0.0)
sn1 = np.nan_to_num(sn1_data, nan=0.0)

# --- Align SN2 → SN3 ---
tform_sn2, _ = aa.find_transform(sn2, ref)
sn2_aligned, _ = aa.apply_transform(tform_sn2, sn2_data, ref_data, fill_value=np.nan)

# --- Align SN1 → SN3 ---
tform_sn1, _ = aa.find_transform(sn1, ref)
sn1_aligned, _ = aa.apply_transform(tform_sn1, sn1_data, ref_data, fill_value=np.nan)

# Save results using SN3 WCS
fits.PrimaryHDU(sn2_aligned, header=ref_header).writeto(
    "../M33-Maps-Calibrated/M33-F9/M33F9-SN2Continuum.fits", overwrite=True
)
fits.PrimaryHDU(sn1_aligned, header=ref_header).writeto(
    "../M33-Maps-Calibrated/M33-F9/M33F9-SN1Continuum.fits", overwrite=True
)

# Initial Alignment

In [ ]:
# Just align based on H alpha map, then use the same WCS for all maps in that field
not_aligned=False
if not_aligned:
    def reproject_fits(input_fits, output_fits, target_wcs):
        """
        Reproject a FITS file to a new WCS and save the result to a new FITS file.

        Parameters:
        - input_fits: str, path to the input FITS file.
        - output_fits: str, path to save the reprojected FITS file.
        - target_wcs: WCS object, the target WCS to reproject to.
        """
        # Open the input FITS file
        with fits.open(input_fits) as hdul:
            data = hdul[0].data
            header = hdul[0].header

        # Reproject the data
        reprojected_data, footprint = reproject_interp((data, header), target_wcs)

        # Create a new FITS file with the reprojected data
        hdu = fits.PrimaryHDU(data=reprojected_data, header=target_wcs.to_header())
        hdu.writeto(output_fits, overwrite=True)


    for field in ['F7', 'F8', 'F9', 'NW', 'NE', 'SW', 'SE']:
        directory = f'M33-Maps-NOT-aligned/M33-{field}'

        #define h alpha map to be the target
        target_wcs = WCS(fits.open(f'{directory}/M33{field}-Haflux.fits')[0].header)

        #use glob to find all the fits files in the directory
        fits_files = glob.glob(f'{directory}/*.fits')

        #create directory for the reprojected files
        output_directory = f'M33-Maps/M33-{field}/'
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)
            print('Made directory:', output_directory)

        # Loop through the FITS files and reproject them
        for file in fits_files:
            input_file = file
            output_file = output_directory+file[28:]
            # Reproject the FITS file
            reproject_fits(input_file, output_file, target_wcs)
            print(f'Reprojected {input_file} to {output_file}')
        


# Flux Calibration 

In [ ]:
not_calibrated=False

if not_calibrated:
    calibrations = {'NE': [0.5037, 1.1034, 1.0652],
                    'NW': [0.6845, 1.0958, 1.0966],
                    'SE': [0.8600, 1.1489, 1.3555],
                    'SW': [0.9279, 1.1128, 0.8805],
                    'F5': [0.6194, 1.5244, 1.5353],
                    'F6': [0.7183, 1.1947, 1.5717],
                    'F7': [0.8184, 0.9557, 1.1887],
                    'F8': [0.9931, 0.8587, 0.9162],
                    'F9': [0.8288, 0.8131, 0.8942]}

    SN1_maps = ['OII3727flux']
    SN2_maps = ['OIII5007flux', 'Hbflux']
    SN3_maps = ['Haflux', 'SII6716flux', 'SII6731flux', 'NII6584flux']

    #save all the calibrated maps in a new folder
    for field in calibrations.keys():
        folder = f'../M33-Maps-Calibrated/M33-{field}'
        # #make the folder if it doesn't exist
        # if not os.path.exists(folder):
        #     os.makedirs(folder)
        for map in SN1_maps:
            file = f'../M33-Maps/M33-{field}/M33{field}-{map}.fits'
            #load the map with astropy
            with fits.open(file) as hdul:
                data = hdul[0].data
                header = hdul[0].header
                wcs = WCS(header)
            #calibrate the map
            calibrated_data = data / calibrations[field][0]
            #save the calibrated map using astropy fits file
            hdu = fits.PrimaryHDU(calibrated_data, header=header)
            hdu.writeto(os.path.join(folder, f'M33{field}-{map}.fits'), overwrite=True)
                    
        for map in SN2_maps:
            file = f'../M33-Maps/M33-{field}/M33{field}-{map}.fits'
            #load the map with astropy
            with fits.open(file) as hdul:
                data = hdul[0].data
                header = hdul[0].header
                wcs = WCS(header)
            #calibrate the map
            calibrated_data = data / calibrations[field][1]
            #save the calibrated map using astropy fits file
            hdu = fits.PrimaryHDU(calibrated_data, header=header)
            hdu.writeto(os.path.join(folder, f'M33{field}-{map}.fits'), overwrite=True)
            
        for map in SN3_maps:
            file = f'../M33-Maps/M33-{field}/M33{field}-{map}.fits'
            #load the map with astropy
            with fits.open(file) as hdul:
                data = hdul[0].data
                header = hdul[0].header
                wcs = WCS(header)
            #calibrate the map
            calibrated_data = data / calibrations[field][2]
            #save the calibrated map using astropy fits file
            hdu = fits.PrimaryHDU(calibrated_data, header=header)
            hdu.writeto(os.path.join(folder, f'M33{field}-{map}.fits'), overwrite=True)